# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mfaiqdev/MLinternship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Data Loading

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("Number of clients:", df["client_hash_id"].nunique())

if "content_hash_id" in df.columns:
    print("Number of content items:", df["content_hash_id"].nunique())

Dataset loaded successfully!
Shape: (9841378, 31)
Number of clients: 55
Number of content items: 331437


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1

The FlyRank research compares AI-search visibility with traditional search performance and shows that the two can provide different signals. This suggests that strong performance in traditional search should not automatically be treated as proof of strong AI-search visibility.

**Methodology question:** Where does the label come from? If the label is based on observed AI referrals or AI visibility, the validation should use an independent signal rather than a rule that directly defines the outcome.

**Validation concern:** A validation design can support a directional relationship, but it should not be interpreted as proof that one search signal directly causes another.

### Finding 2

The research also reports differences in performance across content and search characteristics. This suggests that content-level signals can vary substantially between pages and clients.

**Methodology question:** Are the observations independent across clients and content items? If records from the same client appear in both training and testing data, the model may learn client-specific patterns.

**Validation concern:** A client-grouped split is more appropriate when the intended question is whether the model can generalize to unseen clients.

Overall, the research findings are useful as directional observations, but the strength of the claim depends on how labels are constructed, how independent the validation data are, and whether the evaluation design matches the research question.

In [2]:
# Basic methodology checks for the dataset used in this audit.

required_columns = [
    "client_hash_id",
    "gsc_avg_position",
    "gsc_impressions",
    "gsc_clicks",
    "ga4_pageviews",
    "ga4_users",
    "ga4_total_engagement_sec",
    "scroll_events",
    "ga4_sessions"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing required columns:", missing_columns)
print("Number of clients:", df["client_hash_id"].nunique())

if "content_hash_id" in df.columns:
    print("Number of content items:", df["content_hash_id"].nunique())

Missing required columns: []
Number of clients: 55
Number of content items: 331437


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, the Random Forest was evaluated using a client-grouped split.

For this audit, I kept the same five model features and the same proxy label, while using a controlled training sample so the validation could run within the available memory.

The split remains client-grouped, with 44 clients in training and 11 clients in testing and no client overlap.

The Week-5 Random Forest measured:

- Precision: 0.309
- Recall: 0.693
- F1: 0.428

Under the ML-09 validation setup, the Random Forest measured:

- Precision: 0.122
- Recall: 0.973
- F1: 0.217

The results are not stable across the two runs. Recall increased substantially, but precision decreased and F1 fell from 0.428 to 0.217.

This suggests that the model's performance is sensitive to the training setup. The ML-09 result should therefore be treated as a validation check rather than evidence of a reliable production model.

### Grouped Split

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score

features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_users",
    "ga4_total_engagement_sec",
    "scroll_events"
]

model_df = df.copy()

model_df["refresh_label"] = (
    (model_df["gsc_impressions"] > 100)
    &
    (model_df["gsc_clicks"] <= 1)
    &
    (model_df["ga4_sessions"].fillna(0) == 0)
).astype("int8")

X = model_df[features].fillna(0)
y = model_df["refresh_label"]
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train_full = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train_full = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Full training rows:", len(X_train_full))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx].unique())
        &
        set(groups.iloc[test_idx].unique())
    )
)

Full training rows: 8935676
Test rows: 905702
Training clients: 44
Test clients: 11
Client overlap: 0


### Controlled Sample

In [4]:
# Use a controlled sample to avoid exhausting notebook RAM.
# Keep the test set unchanged.

rng = np.random.RandomState(42)

MAX_TRAIN_ROWS = 500_000

if len(X_train_full) > MAX_TRAIN_ROWS:
    sampled_idx = rng.choice(
        len(X_train_full),
        size=MAX_TRAIN_ROWS,
        replace=False
    )

    X_train = X_train_full.iloc[sampled_idx]
    y_train = y_train_full.iloc[sampled_idx]
else:
    X_train = X_train_full
    y_train = y_train_full

print("Training sample:", X_train.shape)
print("Positive labels:", int(y_train.sum()))
print("Positive rate:", round(y_train.mean(), 4))

Training sample: (500000, 5)
Positive labels: 19843
Positive rate: 0.0397


### Random Forest Model Training

In [5]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=30,
    random_state=42,
    class_weight="balanced",
    n_jobs=2,
    max_depth=12,
    min_samples_leaf=10
)

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

print("Model trained successfully.")

Model trained successfully.


### Comparison

In [6]:
new_precision = precision_score(
    y_test,
    model_predictions,
    zero_division=0
)

new_recall = recall_score(
    y_test,
    model_predictions,
    zero_division=0
)

new_f1 = f1_score(
    y_test,
    model_predictions,
    zero_division=0
)

comparison = pd.DataFrame({
    "Approach": [
        "Week-5 Random Forest",
        "ML-09 Random Forest"
    ],
    "Precision": [
        0.309382,
        new_precision
    ],
    "Recall": [
        0.692788,
        new_recall
    ],
    "F1": [
        0.427744,
        new_f1
    ]
})

comparison

,Approach,Precision,Recall,F1
0,Week-5 Random Forest,0.309382,0.692788,0.427744
1,ML-09 Random Forest,0.121815,0.973474,0.216535


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I checked the final feature set against the proxy label definition.

The proxy label is directly created from:

- `gsc_impressions`
- `gsc_clicks`
- `ga4_sessions`

None of these variables are included as model features.

The final model features are:

- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_users`
- `ga4_total_engagement_sec`
- `scroll_events`

Therefore, there is no direct feature-to-label overlap in the final feature set.

However, `gsc_avg_position` is still closely related to search performance, so its high importance should not be interpreted as causal evidence. It is better described as the strongest measured predictive signal among the selected features.

### Direct feature-to-label leakage check

In [7]:
label_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions"
]

print("Final model features:")
print(features)

print("\nLabel-defining columns:")
print(label_columns)

direct_overlap = set(features).intersection(label_columns)

print("\nDirect feature/label overlap:", direct_overlap)

if len(direct_overlap) == 0:
    print("Leakage check: No direct feature-to-label overlap found.")
else:
    print("Leakage warning: Some label-defining columns are model features.")

Final model features:
['gsc_avg_position', 'ga4_pageviews', 'ga4_users', 'ga4_total_engagement_sec', 'scroll_events']

Label-defining columns:
['gsc_impressions', 'gsc_clicks', 'ga4_sessions']

Direct feature/label overlap: set()
Leakage check: No direct feature-to-label overlap found.


### Client overlap check

In [8]:
train_clients = set(
    groups.iloc[train_idx].unique()
)

test_clients = set(
    groups.iloc[test_idx].unique()
)

client_overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Overlapping clients:", len(client_overlap))

if len(client_overlap) == 0:
    print("Grouped split check: PASS")
else:
    print("Grouped split check: WARNING")

Training clients: 44
Test clients: 11
Overlapping clients: 0
Grouped split check: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original bold claim

"The Random Forest model can identify content that requires refresh attention."

### Safer rewritten claim

The Random Forest measured a directional relationship between the selected ranking and behavioural signals and the refresh proxy label. Under the ML-09 client-grouped validation setup, the model achieved high recall but lower precision, resulting in an F1-score of 0.217.

The result suggests that the selected signals can provide a directional decision-support signal for identifying possible refresh candidates, but the model also produced a substantial number of false positives.

Because the validation used a controlled training sample and a smaller Random Forest than the Week-5 model, the result should not be treated as proof of production-level performance.

Further validation with larger training data, repeated grouped splits, and an independently defined outcome would be needed before making a stronger claim.

In [9]:
# Record the final validation result used in the claim rewrite.

final_result = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1"
    ],
    "ML-09 Random Forest": [
        new_precision,
        new_recall,
        new_f1
    ]
})

print("Final ML-09 validation results:")
display(final_result)

print(
    f"\nThe ML-09 Random Forest measured "
    f"{new_precision:.3f} precision, "
    f"{new_recall:.3f} recall, and "
    f"{new_f1:.3f} F1."
)

print(
    "\nClaim check: Results are presented as measured and directional "
    "decision-support, not as proof of production performance."
)

Final ML-09 validation results:


,Metric,ML-09 Random Forest
0,Precision,0.121815
1,Recall,0.973474
2,F1,0.216535



The ML-09 Random Forest measured 0.122 precision, 0.973 recall, and 0.217 F1.

Claim check: Results are presented as measured and directional decision-support, not as proof of production performance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.